# Module 11 · 5：影片下游 — VideoMAE 動作辨識微調

> 影片資料（M09 影片小節）整理好後，最常見的下游是**動作辨識**：
> 把預訓練 VideoMAE 微調到你自己的動作類別。

## 學習目標
- 複習影片推論流程（抽樣 → processor → 模型）。
- 認識 **VideoMAE 微調**的資料格式 `(N,T,C,H,W)` + 標籤，與 `Trainer` 設定。

<!-- concept-image:05_video_downstream_fig1 -->
<div align="center">
<img src="concept_images/05_video_downstream_fig1_preprocessing_20260611_225021.png" alt="影片資料前處理與特徵提取" width="640" style="max-width:100%;">
<br><sub>圖 1 · 影片資料前處理與特徵提取</sub>
</div>

<!-- concept-image:05_video_downstream_fig2 -->
<div align="center">
<img src="concept_images/05_video_downstream_fig2_videomae_inference_20260611_225108.png" alt="VideoMAE 推論與預訓練" width="640" style="max-width:100%;">
<br><sub>圖 2 · VideoMAE 推論與預訓練</sub>
</div>

<!-- concept-image:05_video_downstream_fig3 -->
<div align="center">
<img src="concept_images/05_video_downstream_fig3_videomae_finetune_20260611_225156.png" alt="影片微調完整管線" width="640" style="max-width:100%;">
<br><sub>圖 3 · 影片微調完整管線</sub>
</div>

In [ ]:
import numpy as np
try:
    import torch
    HAS = True
except Exception:
    HAS = False
    print("需 `uv sync --extra multimodal --extra train`。說明仍可閱讀。")

## A. 推論複習（zero-extra-training）

用在 Kinetics-400 上微調過的 VideoMAE，可直接辨識 400 種動作（見 M09 影片 03）。

In [ ]:
if HAS:
    try:
        import av
        from huggingface_hub import hf_hub_download
        from transformers import AutoImageProcessor, VideoMAEForVideoClassification
        # 真實影片，均勻抽樣 16 影格
        path = hf_hub_download(repo_id="nielsr/video-demo",
                               filename="eating_spaghetti.mp4", repo_type="dataset")
        container = av.open(path)
        allf = [f.to_ndarray(format="rgb24") for f in container.decode(video=0)]
        container.close()
        idx = np.linspace(0, len(allf) - 1, 16).astype(int)
        clip = [allf[i] for i in idx]

        proc = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base-finetuned-kinetics")
        model = VideoMAEForVideoClassification.from_pretrained("MCG-NJU/videomae-base-finetuned-kinetics")
        inputs = proc(clip, return_tensors="pt")
        with torch.no_grad():
            logits = model(**inputs).logits
        print("輸入:", tuple(inputs["pixel_values"].shape), "(N,T,C,H,W)")
        print("預測動作:", model.config.id2label[int(logits.argmax(-1))], "（真實影片）")
    except Exception as e:
        print("（未能下載 VideoMAE，略過）", e)

## B. 微調到自己的動作類別

資料格式：每筆 = 一段 clip 的 16 張抽樣影格 `(T,C,H,W)` + 動作標籤。
換成 `VideoMAEForVideoClassification(num_labels=你的類別數)` 後，用同一套 `Trainer`。

In [ ]:
if HAS:
    print("""微調設定骨架（概念）：

    from transformers import (VideoMAEImageProcessor,
        VideoMAEForVideoClassification, TrainingArguments, Trainer)

    model = VideoMAEForVideoClassification.from_pretrained(
        "MCG-NJU/videomae-base",            # 用未微調的 base 當起點
        num_labels=NUM_ACTIONS,
        ignore_mismatched_sizes=True)

    # dataset: 每筆 = 16 張抽樣影格(經 processor) + 標籤
    args = TrainingArguments(output_dir="...", num_train_epochs=5,
                             per_device_train_batch_size=2, learning_rate=5e-5)
    Trainer(model=model, args=args, train_dataset=ds).train()
    """)
    print("→ 影片微調算力較重（5 維張量），實務多用 GPU + 少量影格 + LoRA 等技巧。")

## 小結

| 階段 | 資料 | 模型 |
|:--|:--|:--|
| 推論 | `(N,T,C,H,W)` | VideoMAE (Kinetics 微調版) |
| 微調 | `(N,T,C,H,W)` + 動作標籤 | VideoMAE base + `Trainer` |

影片是四模態中算力最重的；資料前處理（抽樣！）對效率影響最大。